# HALO-AC3 Campaign Data Analysis

Radiative transfer simulation using atmospheric measurements from the HALO-AC3 Arctic campaign. Validation of simulated albedo against aircraft observations over Arctic sea ice.

In [1]:
# Import PyRadtran components
from pyradtran.config import PathsConfig, SimulationDefaults, SimulationConfig, ExecutionConfig, OutputConfig, load_config
from pyradtran.core import Simulation, generate_input_content
from pyradtran.io import parse_uvspec_output
from pyradtran.interface import PyRadtranAccessor  # This should register the accessor automatically
from dataclasses import asdict

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from pathlib import Path
import os
import yaml
import pandas as pd

# Load the configuration from the YAML file
config_path = Path('config/HALO-AC3_HALO_solar_disort.yaml')
import logging
# Configure logging for pyradtran
logging.getLogger('pyradtran').setLevel(logging.CRITICAL)

In [2]:
ds = xr.open_dataset('data/HALO-AC3_HALO_aircraft_broadband_radiation_clear_sky_with_ocean_600s.nc').interpolate_na('time')

# the albedo has over 30% missing values, so we interpolate it for now


/projekt_agmwend/home_rad/Joshua/micromamba/envs/mamba_josh/lib/python3.12/site-packages/xarray/backends/plugins.py:110: RuntimeWarning: Engine 'gini' loading failed:
module 'numpy' has no attribute 'cumproduct'
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)


In [3]:
# Run a spectral simulation for the dataset
print("Running batch spectral simulation...")
ds_sim = ds.pyradtran.run_uvspec(
    config_path=config_path,
    return_dataset=True,
    save_to_file=True,
    output_path='data/Simulated_HALO-AC3_HALO_aircraft_broadband_radiation_clear_sky_with_ocean_600s.nc',
    albedo_var='albedo',
)

print("\nSimulation complete!")
ds_sim

Running batch spectral simulation...


FileNotFoundError: Configuration file not found: config/HALO-AC3_HALO_solar_disort.yaml

## Simulation Results

Spectral radiative transfer calculations completed using DISORT solver with multi-level output altitude grid.

In [4]:
albedo_sim_z0 = ds_sim.albedo.isel(altitude=0)
albedo_sim_z10 = ds_sim.albedo.isel(altitude=10)
albedo_meas = ds.albedo

fig, (ax, ax_scatter) = plt.subplots(1, 2, gridspec_kw={'width_ratios': [2, 1]}, figsize=(12, 4))
ax.plot(albedo_sim_z10, label='Simulated Albedo at 10m', linestyle='-', marker='x', color='black', alpha=0.7)
ax.plot(albedo_meas, label='Measured Albedo', linestyle='--', marker='o', color='blue', alpha=0.7)
ax.plot(albedo_sim_z0, label='Simulated Albedo at 0m', linestyle='-', marker='s', color='green', alpha=0.7  )
ax.grid(alpha=0.3)
ax.set_xlabel('Time')
ax.set_ylabel('Albedo')
ax.legend(loc='upper right')

ax_scatter.scatter(albedo_meas, albedo_sim_z0, label='sim zout 0m', color='green', alpha=0.5)
ax_scatter.scatter(albedo_meas, albedo_sim_z10, label='sim zout 10m', color='black', alpha=0.5)
ax_scatter.plot([0, 1], [0, 1], linestyle='--', color='k', alpha=0.5)
ax_scatter.set_xlabel('Measured Albedo')
ax_scatter.set_ylabel('Simulated Albedo')
ax_scatter.legend(loc='upper left')


NameError: name 'ds_sim' is not defined